In [ ]:
import pandas as pd
from datetime import datetime

# =====================================================
# Case Study 2: Predictive Workforce Planning (CV Delivery Forecasting)
# Python Script for Prediction, Methodology & Justification
# =====================================================

# Step 1: Load the historical data
print("Loading data...")
df = pd.read_csv("/content/sample_data/CV OCD Data Sep to Jan.csv")

# Step 2: Filter only CV deliveries (ignore OCD)
df_cv = df[df["CV /OCD"] == "CV"].copy()

# Step 3: Parse dates (format is DD/M/YYYY or DD/MM/YYYY → dayfirst=True)
df_cv["CV Delivery Date"] = pd.to_datetime(df_cv["CV Delivery Date"], dayfirst=True, errors="coerce")
df_cv = df_cv.dropna(subset=["CV Delivery Date"])

# Step 4: Compute daily CV delivery counts from historical data
daily_cv = df_cv.groupby("CV Delivery Date").size().reset_index(name="cv_deliveries")
daily_cv = daily_cv.set_index("CV Delivery Date").sort_index()

# Create a complete daily index (fill missing days with 0 deliveries)
start_date = daily_cv.index.min()
end_date = daily_cv.index.max()
full_dates = pd.date_range(start=start_date, end=end_date, freq="D")
daily_cv = daily_cv.reindex(full_dates, fill_value=0)

# IMPORTANT: Use ONLY data up to 31-Jan-2026 as historical (avoid any data leakage from Feb onwards)
daily_cv = daily_cv[daily_cv.index < pd.to_datetime("2026-02-01")]

print(f"Historical period used: {daily_cv.index.min().date()} to {daily_cv.index.max().date()}")
print(f"Total historical days: {len(daily_cv)}")
print(f"Average daily CV deliveries: {daily_cv['cv_deliveries'].mean():.2f}")

# Step 5: Chosen Model → Day-of-Week Average (Seasonal Naive Forecast)
# This directly addresses the "Weekly Seasonality" key factor mentioned in the case study.
daily_cv["day_of_week"] = daily_cv.index.day_name()
avg_by_dow = daily_cv.groupby("day_of_week")["cv_deliveries"].mean().round(2)

print("\nAverage CV deliveries by day of week (from 5 months historical data):")
print(avg_by_dow)

# Step 6: Predict first 10 days of February 2026
prediction_dates = pd.date_range(start="2026-02-01", end="2026-02-10", freq="D")
prediction_df = pd.DataFrame(index=prediction_dates)
prediction_df["day_of_week"] = prediction_df.index.day_name()

# Map the historical day-of-week average to each prediction day
prediction_df["forecasted_cv_deliveries"] = prediction_df["day_of_week"].map(avg_by_dow).round(0).astype(int)

print("\n" + "="*60)
print("PREDICTION FOR FIRST 10 DAYS OF FEBRUARY 2026")
print("="*60)
print(prediction_df[["day_of_week", "forecasted_cv_deliveries"]])

# Step 7: Save results (for your Working Files deliverable)
prediction_df[["forecasted_cv_deliveries"]].to_csv("CV_Delivery_Forecast_Feb2026.csv")
print("\nForecast saved to: CV_Delivery_Forecast_Feb2026.csv")

# Optional: Also save the full historical daily series for reference
daily_cv[["cv_deliveries"]].to_csv("historical_daily_cv_deliveries.csv")

print("\n" + "="*60)
print("METHODOLOGY & JUSTIFICATION (copy-paste into your PDF report)")
print("="*60)
print("""Chosen Model: Day-of-Week Average (Seasonal Naive Forecast)

Technical Note:
- Extracted daily CV delivery counts from the CSV (filtered CV /OCD == 'CV').
- Created a complete daily time series and restricted it to data before 2026-02-01.
- Computed the average number of CV deliveries for each day of the week across the entire 5-month historical period.
- For the forecast period (1–10 Feb 2026), assigned the historical average corresponding to that day’s weekday.

Why this method?
- Directly incorporates the **Weekly Seasonality** factor explicitly mentioned in the case study.
- Extremely simple, interpretable, and requires no hyper-parameter tuning (perfect for operational use by Terminal Manager).
- Implicitly captures the **4-Day Rule** and **Yard Density Factor** because these effects are already reflected in the historical delivery volumes.
- Moving Average / Linear Regression / complex Time-Series ML models were considered, but a day-of-week average is more robust here given the strong weekly pattern and limited need for trend extrapolation over just 10 days.
- If more data/features were available (e.g., exact yard inventory on 31-Jan-2026 or daily imports in late Jan), a regression model with lags could be added.

Assumptions made (as per note in case study):
- The weekly delivery pattern observed in Sep 2025 – Jan 2026 continues into early Feb 2026.
- No major external disruptions (holidays, vessel delays, etc.) in the first 10 days of Feb.
- The February import numbers provided in the scenario were noted but not directly used, as deliveries in early Feb are predominantly driven by prior inventory (the 4-Day Rule effect is already baked into historical averages).

""")

Loading data...
Historical period used: 2025-08-12 to 2026-01-31
Total historical days: 173
Average daily CV deliveries: 65.29

Average CV deliveries by day of week (from 5 months historical data):
day_of_week
Friday       68.04
Monday       72.79
Saturday     58.60
Sunday       30.42
Thursday     84.24
Tuesday      76.00
Wednesday    65.88
Name: cv_deliveries, dtype: float64

PREDICTION FOR FIRST 10 DAYS OF FEBRUARY 2026
           day_of_week  forecasted_cv_deliveries
2026-02-01      Sunday                        30
2026-02-02      Monday                        73
2026-02-03     Tuesday                        76
2026-02-04   Wednesday                        66
2026-02-05    Thursday                        84
2026-02-06      Friday                        68
2026-02-07    Saturday                        59
2026-02-08      Sunday                        30
2026-02-09      Monday                        73
2026-02-10     Tuesday                        76

Forecast saved to: CV_Delivery_For